# Music Recommender API worker

Create a **Worker API** key in the Music Recommender UI, then run the cell below. Python 3.11 is recommended; on Python 3.12+ the cell applies a packaging-only compatibility patch for the pinned OpenL3/resampy source releases without changing model code or versions. The cell performs a one-track dry run by default; exit code 2 is expected when unembedded tracks remain. Remove `--dry-run` and `--limit 1` only when you are ready to write embeddings. The key is prompted for rather than stored in notebook source.

In [ ]:
import getpass
import os
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("The worker requires Python 3.11 or newer.")

repo_url = "https://github.com/poesterlin/music-recommender.git"
repo_dir = "music-recommender"
if not os.path.isdir(os.path.join(repo_dir, "embeddings")):
    subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
else:
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
os.chdir(repo_dir)
if sys.version_info >= (3, 12):
    subprocess.run([sys.executable, "embeddings/install_python312.py"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "embeddings/requirements.txt"], check=True)

os.environ["WORKER_URL"] = "https://music.oesterlin.dev"
os.environ["WORKER_TOKEN"] = getpass.getpass("Paste worker API key: ")
completed = subprocess.run([
    sys.executable,
    "embeddings/worker.py",
    "--source-mode",
    "api",
    "--duration",
    "60",
    "--dry-run",
    "--limit",
    "1",
])
if completed.returncode == 2:
    print("Dry run completed; pending tracks remain (expected exit code 2).")
elif completed.returncode != 0:
    raise subprocess.CalledProcessError(completed.returncode, completed.args)